In [4]:
# =============== 核心：强制绕过所有代理（解决你的网络问题）===============
import os
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["NO_PROXY"] = "*"
# ======================================================================

# 自动安装依赖（如果没装的话）
try:
    import baostock as bs
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "baostock", "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"])
    import baostock as bs

import pandas as pd
from datetime import datetime, timedelta

# ===================== 配置（和你之前完全一致）=====================
STOCKS = {
    "sz.002460": {"name": "赣锋锂业", "industry": "矿产资源类"},
    "sh.600733": {"name": "北汽蓝谷", "industry": "制造业类"},
    "sz.300182": {"name": "捷成股份", "industry": "AI广告类"}
}
# 直接覆盖你之前的 stock_data 文件夹
SAVE_DIR = "stock_data"
os.makedirs(SAVE_DIR, exist_ok=True)

# 时间范围：近1年（和你之前的模拟数据时间完全一致）
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")

# ===================== 自动重试登录（解决网络波动）=====================
def login_baostock(max_retries=3):
    for i in range(max_retries):
        lg = bs.login()
        if lg.error_code == '0':
            print(f"✅ Baostock 登录成功（第{i+1}次尝试）")
            return True
        print(f"⚠️  第{i+1}次登录失败，重试中...")
    print("❌ Baostock 登录失败，请检查网络")
    return False

# ===================== 主程序 =====================
print("="*60)
print("📊 开始获取真实A股数据（Baostock 数据源）")
print(f"📅 时间范围: {start_date} 至 {end_date}")
print(f"📈 目标股票: {[v['name'] for v in STOCKS.values()]}")
print("="*60)

if not login_baostock():
    exit()

all_basic = []

for bs_code, info in STOCKS.items():
    code = bs_code.split(".")[1]
    name = info["name"]
    print(f"\n正在处理: {name} ({code})")
    
    # 1. 获取日K线数据（前复权，最适合分析）
    rs = bs.query_history_k_data_plus(
        bs_code,
        "date,open,high,low,close,volume",
        start_date=start_date,
        end_date=end_date,
        frequency="d",
        adjustflag="2"  # 3=前复权，1=后复权，2=不复权
    )
    
    # 2. 转换为DataFrame并处理数据
    df = rs.get_data()
    
    # 转换数据类型（确保和模拟数据格式完全一致）
    df["open"] = df["open"].astype(float).round(2)
    df["close"] = df["close"].astype(float).round(2)
    df["high"] = df["high"].astype(float).round(2)
    df["low"] = df["low"].astype(float).round(2)
    df["volume"] = df["volume"].astype(int)
    
    # 添加股票标识（和模拟数据列顺序完全一致）
    df["stock_code"] = code
    df["stock_name"] = name
    
    # 3. 保存历史交易数据（覆盖之前的模拟文件）
    history_path = os.path.join(SAVE_DIR, f"{code}_{name}_历史交易数据.csv")
    df.to_csv(history_path, index=False, encoding="utf-8-sig")
    print(f"  ✅ 历史数据已保存：{len(df)} 条真实记录")
    
    # 4. 生成并保存基本信息
    basic_df = pd.DataFrame([{
        "股票代码": code,
        "股票名称": name,                                                                                                               
        "所属行业": info["industry"],
        "数据更新时间": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "数据来源": "Baostock 真实A股数据（前复权）"
    }])
    basic_path = os.path.join(SAVE_DIR, f"{code}_{name}_基本信息.csv")
    basic_df.to_csv(basic_path, index=False, encoding="utf-8-sig")
    all_basic.append(basic_df)
    print(f"  ✅ 基本信息已保存")

# 5. 生成所有股票基本信息汇总
pd.concat(all_basic).to_csv(os.path.join(SAVE_DIR, "所有股票基本信息汇总.csv"), index=False, encoding="utf-8-sig")

# 登出Baostock
bs.logout()

print("\n" + "="*60)
print("🎉 全部真实数据获取完成！")
print(f"📂 文件保存路径：{os.path.abspath(SAVE_DIR)}")
print("✅ 已自动覆盖之前的模拟数据，格式完全一致")
print("✅ 开代理也能正常使用，无需关闭")
print("="*60)

📊 开始获取真实A股数据（Baostock 数据源）
📅 时间范围: 2025-04-28 至 2026-04-28
📈 目标股票: ['赣锋锂业', '北汽蓝谷', '捷成股份']
login success!
✅ Baostock 登录成功（第1次尝试）

正在处理: 赣锋锂业 (002460)
  ✅ 历史数据已保存：242 条真实记录
  ✅ 基本信息已保存

正在处理: 北汽蓝谷 (600733)
  ✅ 历史数据已保存：242 条真实记录
  ✅ 基本信息已保存

正在处理: 捷成股份 (300182)
  ✅ 历史数据已保存：242 条真实记录
  ✅ 基本信息已保存
logout success!

🎉 全部真实数据获取完成！
📂 文件保存路径：d:\vscode\project\task8\stock_data
✅ 已自动覆盖之前的模拟数据，格式完全一致
✅ 开代理也能正常使用，无需关闭
